In [57]:
from game import Game
import time
import os
import re
from collections import namedtuple
from collections import deque
import copy  
from heapq import heappush, heappop
import pandas as pd

## Load maps

In [3]:
def extract_map_files(directory):
    pattern = re.compile(r'^map(\d+)\.txt$')
    map_file_indices = []

    for file_name in os.listdir(directory):
        match = pattern.match(file_name)
        if match:
            map_file_indices.append(match.group(1))

    return [int(idx) for idx in map_file_indices]

def is_valid_input(map, indices, algorithm, solvers):
    valid_input = True
    if map not in indices:
        print(f"Map index out of range. Please choose within range {min(indices)} to {max(indices)}")
        valid_input = False
    if algorithm not in solvers.keys():    
        print(f"{algorithm} is not a defined algorithm. Please choose from", *[f"{solver} ({i+1})  " for i, solver in enumerate(solvers.keys())])
        valid_input = False
    return valid_input

def load_map(map_index):  
    file_name = "map" + str(map_index) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    return game_map

map_file_indices = extract_map_files("./assets/maps/")

## Tutorial

In [ ]:
print("This is an example of the game map:")
map = load_map(2)
game = Game(map)
game.display_map()

This is an example of the game map:
W	P1	H	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [5]:
game.get_box_locations()

[(2, 3), (3, 2), (4, 3)]

In [6]:
game.get_goal_locations()

[(1, 3), (3, 1), (5, 3)]

In [7]:
game.get_player_position()

(0, 2)

- W : Wall
- H : Human
- B : Box
- P : Portal
- G : Goal

In [8]:
for direction in ['U', 'D', 'R', 'L']:
    result = game.apply_move(direction)
    print(f"Move {direction} is valid: {result}")
    if result:
        game.display_map()

Move U is valid: False
Move D is valid: False
Move R is valid: False
Move L is valid: True
W	P1	.	W	W	W	W
W	W	W	G1	W	W	W
W	W	W	B1	W	W	W
W	G2	B2	H	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [9]:
game.apply_move('U')
game.display_map()

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	H	W	W	W
W	G2	B2	.	P1	W	W
W	W	W	B3	W	W	W
W	W	W	G3	W	W	W
W	W	W	W	W	W	W


In [10]:
game.apply_moves(['D', 'L', 'R', 'D']) 
game.display_map()
print("Is game won?", game.is_game_won())

W	P1	.	W	W	W	W
W	W	W	G1/B1	W	W	W
W	W	W	.	W	W	W
W	G2/B2	.	.	P1	W	W
W	W	W	H	W	W	W
W	W	W	G3/B3	W	W	W
W	W	W	W	W	W	W
Is game won? True


## Solvers

In [4]:
def find_solution(final_position, game):
    path = []
    current = final_position
    while current.parent is not None:
        path.append(get_move(current.parent, current, game)) 
        current = current.parent
    return path[::-1]

def get_move(prev_state, current_state, game):
    game.set_player_position(prev_state.player_position)
    game.set_box_positions(prev_state.box_position)
    for move in ['U', 'D', 'R', 'L']:
        game.apply_move(move)
        if game.get_player_position() == current_state.player_position and game.get_box_locations() == list(current_state.box_position):
            return move
        game.set_player_position(prev_state.player_position)
        game.set_box_positions(prev_state.box_position)
    return ""

### BFS

In [42]:
# TODO: Must return moves (if there is no solution return None), number of visited states

def solver_bfs(game_map):
    game = Game(game_map)
    bfs = deque()
    visited = set()
    Node = namedtuple("Node", ["player_position", "parent", "box_position"])
    
    first_position = Node(player_position=game.get_player_position(), parent=None, box_position=tuple(game.get_box_locations()))
    bfs.append(first_position)
    visited.add((first_position.player_position, first_position.box_position))

    while bfs:  
        current_position = bfs.popleft()
        game.set_player_position(current_position.player_position)
        game.set_box_positions(current_position.box_position)

        if game.is_game_won():
            return find_solution(current_position, game), len(visited)

        for move in ['U', 'L', 'R', 'D']:
            game.set_player_position(current_position.player_position)
            game.set_box_positions(current_position.box_position)

            if game.apply_move(move):
                next_position = Node(player_position=game.get_player_position(),parent=current_position, box_position=tuple(game.get_box_locations()))
                if (next_position.player_position, next_position.box_position) not in visited:
                    visited.add((next_position.player_position, next_position.box_position))
                    bfs.append(next_position)

    return None, 0

### DFS

In [6]:
from collections import deque, namedtuple

def solver_dfs(game_map):
    game = Game(game_map)
    dfs = []
    visited = set()
    Node = namedtuple("Node", ["player_position", "parent", "box_position"])
    
    first_position = Node(player_position=game.get_player_position(), parent=None, box_position=tuple(game.get_box_locations()))
    dfs.append(first_position)
    visited.add((first_position.player_position, first_position.box_position))

    while dfs:
        current_position = dfs.pop()
        game.set_player_position(current_position.player_position)
        game.set_box_positions(current_position.box_position)
        
        if game.is_game_won():
            return find_solution(current_position, game), len(visited)
        
        for move in ['U', 'L', 'R', 'D']:
            game.set_player_position(current_position.player_position)
            game.set_box_positions(current_position.box_position)
            
            if game.apply_move(move):
                next_position = Node(player_position=game.get_player_position(), parent=current_position, box_position=tuple(game.get_box_locations()))
                
                if (next_position.player_position, next_position.box_position) not in visited:
                    visited.add((next_position.player_position, next_position.box_position))
                    dfs.append(next_position)
    
    return None, 0

### IDS

In [7]:
# TODO: Must return moves, number of visited states

def solver_ids(game_map):
    game = Game(game_map)
    max_depth = len(game_map) * len(game_map[0])
    depth = 0
    
    while depth<max_depth:
        game = Game(copy.deepcopy(game_map))
        ids = []
        visited = set()
        Node = namedtuple("Node", ["player_position", "parent", "box_position", "depth"])
        first_position = Node(player_position=game.get_player_position(), parent=None, box_position=tuple(game.get_box_locations()), depth=0)
        ids.append(first_position)
        visited.add((first_position.player_position, first_position.box_position))

        while ids:
            current_position = ids.pop()
            game.set_player_position(current_position.player_position)
            game.set_box_positions(current_position.box_position)

            if game.is_game_won():
                return find_solution(current_position, game), len(visited)
        
            if current_position.depth < depth: 
                for move in ['U', 'L', 'R', 'D']:
                    game.set_player_position(current_position.player_position)
                    game.set_box_positions(current_position.box_position)

                    if game.apply_move(move):
                        next_position = Node(player_position=game.get_player_position(), parent=current_position, box_position=tuple(game.get_box_locations()), depth=current_position.depth + 1)
                        if (next_position.player_position, next_position.box_position) not in visited:
                            visited.add((next_position.player_position, next_position.box_position))
                            ids.append(next_position)
        depth += 1 
    return None, 0

### A*

In [8]:
# TODO: Heuristic function
def heuristic(game):
    player_position = game.get_player_position()
    box_positions = game.get_box_locations()
    goal_positions = game.get_goal_locations()

    player_to_box = min(abs(player_position[0] - bx) + abs(player_position[1] - by) for (bx, by) in box_positions)
    box_to_goal = sum(min(abs(bx - gx) + abs(by - gy) for (gx, gy) in goal_positions) for (bx, by) in box_positions)
    
    return box_to_goal + player_to_box

# TODO: Must return moves, number of visited states
def solver_astar(game_map, heuristic_func=heuristic, weight=1):
    game = Game(game_map)
    astar = []
    visited = {}
    
    Node = namedtuple("Node", ["f_score", "g_score", "player_position", "parent", "box_position"])
    first_position = Node(f_score=0, g_score=0, player_position=game.get_player_position(), parent=None, box_position=tuple(game.get_box_locations())) 
    heappush(astar, (first_position.f_score, first_position))
    visited[first_position] = 0
    
    while astar:
        _, current = heappop(astar)
        
        game.set_player_position(current.player_position)
        game.set_box_positions(current.box_position)

        if game.is_game_won():
            return find_solution(current, game), len(visited)

        for move in ['U', 'L', 'R', 'D']:
            game.set_player_position(current.player_position)
            game.set_box_positions(current.box_position)

            if game.apply_move(move):
                next_position = game.get_player_position()
                next_boxes = tuple(game.get_box_locations())
                g_new = current.g_score + 1
                h_new = heuristic_func(game)
                f_new = g_new + weight * h_new
                next_position = Node(f_score=f_new, g_score=g_new, player_position=game.get_player_position(), parent=current, box_position=tuple(game.get_box_locations()))

                if (next_position.player_position, next_position.box_position) not in visited or g_new < visited[(next_position.player_position, next_position.box_position)]:
                    visited[(next_position.player_position, next_position.box_position)] = g_new
                    heappush(astar, (next_position.f_score, next_position))

    return None, 0

## Weighted A*

In [ ]:
def solver_weighted_astar(game_map, heuristic_func=heuristic, weight=3):
    game = Game(game_map)
    astar = []
    visited = {}
    
    Node = namedtuple("Node", ["f_score", "g_score", "player_position", "parent", "box_position"])
    first_position = Node(f_score=0, g_score=0, player_position=game.get_player_position(), parent=None, box_position=tuple(game.get_box_locations())) 
    heappush(astar, (first_position.f_score, first_position))
    visited[first_position] = 0
    
    while astar:
        _, current = heappop(astar)
        
        game.set_player_position(current.player_position)
        game.set_box_positions(current.box_position)

        if game.is_game_won():
            return find_solution(current, game), len(visited)

        for move in ['U', 'L', 'R', 'D']:
            game.set_player_position(current.player_position)
            game.set_box_positions(current.box_position)

            if game.apply_move(move):
                next_position = game.get_player_position()
                next_boxes = tuple(game.get_box_locations())
                g_new = current.g_score + 1
                h_new = heuristic_func(game)
                f_new = g_new + weight * h_new
                next_position = Node(f_score=f_new, g_score=g_new, player_position=game.get_player_position(), parent=current, box_position=tuple(game.get_box_locations()))

                if (next_position.player_position, next_position.box_position) not in visited or g_new < visited[(next_position.player_position, next_position.box_position)]:
                    visited[(next_position.player_position, next_position.box_position)] = g_new
                    heappush(astar, (next_position.f_score, next_position))

    return None, 0

## Solve

In [78]:
SOLVERS = {
    "BFS": solver_bfs,
    "DFS": solver_dfs,
    "IDS": solver_ids,
     "A*": solver_astar,
     "Weighted_A*": solver_weighted_astar
}

In [79]:
def solve(map, method):  
    
    if not is_valid_input(map, map_file_indices, method, SOLVERS):
        return
    
    file_name = "map" + str(map) + ".txt"
    with open('./assets/maps/' + file_name) as f:
        game_map = f.read()
    
    start_time = time.time()
    moves, numof_visited_states = SOLVERS[method](game_map)
    end_time = time.time()
    print(f"{method} took {round(end_time - start_time, 2)} seconds on map {map} and visited {numof_visited_states} states.")
    
    if moves is None:
        print("No Solution Found!")
    else:
        print(f"{len(moves)} moves were used: {moves}")
            

In [56]:
solve(1, "BFS") # Solve map 1 using BFS

BFS took 0.0 seconds on map 1 and visited 47 states.
7 moves were used: ['U', 'D', 'L', 'R', 'R', 'L', 'D']


In [19]:
def solve_all():
    for map in range(min(map_file_indices), max(map_file_indices) + 1):
        for method in SOLVERS.keys():
            solve(map, method)
            

In [ ]:
solve_all() # Solve all maps using all methods

BFS took 0.0 seconds on map 1 and visited 47 states.
7 moves were used: ['U', 'D', 'L', 'R', 'R', 'L', 'D']
DFS took 0.0 seconds on map 1 and visited 17 states.
7 moves were used: ['D', 'U', 'R', 'L', 'L', 'R', 'U']
IDS took 0.0 seconds on map 1 and visited 17 states.
7 moves were used: ['D', 'U', 'R', 'L', 'L', 'R', 'U']
A* took 0.0 seconds on map 1 and visited 47 states.
7 moves were used: ['D', 'U', 'R', 'L', 'L', 'R', 'U']
BFS took 0.0 seconds on map 2 and visited 26 states.
6 moves were used: ['L', 'U', 'D', 'L', 'R', 'D']
DFS took 0.0 seconds on map 2 and visited 13 states.
6 moves were used: ['L', 'D', 'U', 'L', 'R', 'U']
IDS took 0.0 seconds on map 2 and visited 13 states.
6 moves were used: ['L', 'D', 'U', 'L', 'R', 'U']
A* took 0.0 seconds on map 2 and visited 27 states.
6 moves were used: ['L', 'D', 'U', 'L', 'R', 'U']
BFS took 0.0 seconds on map 3 and visited 130 states.
13 moves were used: ['U', 'L', 'D', 'D', 'U', 'U', 'U', 'U', 'R', 'D', 'D', 'D', 'D']
DFS took 0.0 secon

**Report**

**Question 1**

States can be defined as:

Mike's position(H).


Boxes' position (Bi).


Locations' position (Gi).


Walls' position (W).


Portals' position (Pi).

Main challenge is storing whole data takes up a lot of memory.
We can store position of objects. We can use hashable representation to compare states faster.

**Question 2**

Actions define in four main directions.

U: up       D: down         L: left          R: right


If Mike hits a box, he will push it, if there is an empty space or portal after the box. If the box enters a portal, it will exit the other side of the portal.


Preconditions: The movement must be valid (not into a wall).


Postconditions: The new state of the warehouse is updated accordingly.

**Question 3**

Initial State: The given map configuration.


Goal State: All boxes (Bi) are placed on their goal positions (Gi).


Actions: Moving in four directions while following constraints (walls, boxes, and portals).


Cost Function: Each movement costs the same.

**Question 4**

Heuristic Improvement: Use an admissible heuristic such as the Manhattan Distance to estimate the cost.


Delete Unnecessary States: Avoid revisiting already explored states.


Greedy Expansion: Prioritize states that are closer to the goal.


Weighted A*: Use different weight values w to accelerate the search process.

**Question 5**

BFS:


 Explores all possible moves level by level.


 Strength: Guarantees shortest path.
 Weakness: High memory usage.


 DFS:


Explores one path deeply before backtracking.


Strength: Low memory usage.
Weakness: Might get stuck in deep paths.




 IDS:


Repeatedly runs DFS with increasing depth limits.


Strength: Combines benefits of BFS and DFS.
Weakness: Still computationally expensive.


 A*:


Uses a heuristic (h(n)) to estimate the cost to the goal and expands the most promising path.


Strength: Efficient with heuristics.
Weakness: Depends on heuristic quality.


 Weighted A*:


A modified A* that multiplies the heuristic by a weight (w) to speed up search.


Strength: Speeds up A* by biasing the heuristic.
Weakness: May sacrifice optimality.

**a:**
DFS might get stuck in deep paths. But it's useful in problems where space complexity is a concern.


**b:**
IDS balances between BFS’s guarantee of optimality and DFS’s low memory usage.

**Question 7**

A heuristic function must be:
Admissible: never overestimates cost.
Consistent: obeys the triangle inequality.

**Question 8**

In [ ]:
def Manhattan_heuristic(game):
    player_position = game.get_player_position()
    box_positions = game.get_box_locations()
    goal_positions = game.get_goal_locations()

    box_to_goal = sum(min(abs(bx - gx) + abs(by - gy) for (gx, gy) in goal_positions) for (bx, by) in box_positions)
    
    return box_to_goal 

for i in range(10):
    if i != 8:
        map = load_map(i+1)
        print(solver_weighted_astar(map, heuristic_func=Manhattan_heuristic, weight=3))

(['U', 'D', 'L', 'R', 'R', 'L', 'D'], 23)
(['L', 'U', 'D', 'L', 'R', 'D'], 17)
(['U', 'L', 'D', 'D', 'U', 'U', 'U', 'U', 'R', 'D', 'D', 'D', 'D'], 37)
(None, 0)
(['L', 'U', 'L', 'D', 'L', 'R', 'D', 'R', 'D', 'L', 'L', 'U', 'L', 'U', 'U'], 171)
(['R', 'R', 'D', 'D', 'D', 'D', 'D', 'R', 'L', 'L', 'L', 'L', 'L', 'L', 'L', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'U', 'R', 'U', 'L', 'R', 'R', 'R', 'R', 'R', 'R', 'R'], 2634)
(['R', 'U', 'R', 'R', 'D', 'D', 'D', 'D', 'L', 'D', 'R', 'U', 'U', 'U', 'U', 'L', 'L', 'L', 'R', 'D', 'R', 'D', 'R', 'D', 'D', 'L', 'L', 'D', 'L', 'L', 'U', 'U', 'D', 'R'], 11885)
(['U', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'U', 'R', 'D', 'D', 'R'], 141)
(['R', 'R', 'R', 'R', 'R', 'D', 'R', 'U', 'L', 'U', 'R', 'U', 'L', 'L', 'L', 'U', 'L', 'L', 'D', 'R', 'U', 'U', 'L', 'D', 'R', 'D', 'L', 'D', 'R', 'R', 'D', 'R', 'U', 'L', 'U', 'R', 'U', 'R', 'D', 'D', 'R', 'D', 'L', 'L', 'L', 'L'], 62226)


If computation time is crucial, Manhattan heuristic is better.
But solution quality matters (fewer moves...), the first heuristic should perform better.

**Question 9**

In [222]:
print("For map 1:")
data = {
    "Algorithm": ["BFS", "DFS", "IDS", "A*", "Weighted A*"],
    "Execution Time (seconds)": [0.0, 0.0, 0.0099, 0.004, 0.003],  
    "Visited States": [47, 17, 17, 47, 23],  
    "Solution": ["['U', 'D', 'L', 'R', 'R', 'L', 'D']", "['D', 'U', 'R', 'L', 'L', 'R', 'U']", "['D', 'U', 'R', 'L', 'L', 'R', 'U']", "['D', 'U', 'R', 'L', 'L', 'R', 'U']", "['U', 'D', 'L', 'R', 'R', 'L', 'D']"]  
}

df = pd.DataFrame(data)
print(df)


print("For map 2:")
data = {
    "Algorithm": ["BFS", "DFS", "IDS", "A*", "Weighted A*"],
    "Execution Time (seconds)": [0.0, 0.0, 0.0, 0.0, 0.001],  
    "Visited States": [26, 13, 13, 27, 17],  
    "Solution": ["['L', 'U', 'D', 'L', 'R', 'D']", "['L', 'D', 'U', 'L', 'R', 'U']", "['L', 'D', 'U', 'L', 'R', 'U']", "['L', 'D', 'U', 'L', 'R', 'U']", "['L', 'U', 'D', 'L', 'R', 'D']"]  
}

df = pd.DataFrame(data)
print(df)



print("For map 3:")
data = {
    "Algorithm": ["BFS", "DFS", "IDS", "A*", "Weighted A*"],
    "Execution Time (seconds)": [0.0, 0.0, 0.01, 0.004, 0.004],  
    "Visited States": [130, 38, 38, 95, 30],  
    "Solution": ["['U', 'L', 'D', 'D', 'U', 'U', 'U', 'U', 'R', 'D', 'D', 'D', 'D']", "['U', 'L', 'D', 'D', 'U', 'U', 'U', 'U', 'R', 'D', 'D', 'D', 'D']", "['U', 'L', 'D', 'D', 'U', 'U', 'U', 'U', 'R', 'D', 'D', 'D', 'D']", "['U', 'L', 'U', 'U', 'R', 'D', 'D', 'D', 'D', 'U', 'U', 'L', 'D', 'D']", "['U', 'L', 'D', 'D', 'U', 'U', 'U', 'U', 'R', 'D', 'D', 'D', 'D']"]  
}

df = pd.DataFrame(data)
print(df)




print("For map 4:")
data = {
    "Algorithm": ["BFS", "DFS", "IDS", "A*", "Weighted A*"],
    "Execution Time (seconds)": [0.0, 0.0, 0.0, 0.0, 0.0],  
    "Visited States": [0, 0, 0, 0, 0],  
    "Solution": ["None", "None", "None", "None", "None]"]  
}

df = pd.DataFrame(data)
print(df)



print("For map 5:")
data = {
    "Algorithm": ["BFS", "DFS", "IDS", "A*", "Weighted A*"],
    "Execution Time (seconds)": [0.07, 0.0, 0.2, 0.096, 0.003],  
    "Visited States": [8261, 213, 3016, 1885, 98],  
    "Solution": ["['U', 'L', 'D', 'D', 'R', 'D', 'L', 'L', 'L', 'U', 'U', 'U', 'R', 'U', 'L']", "['D', 'D', 'L', 'L', 'L', 'L', 'U', 'U', 'R', 'D', 'D', 'R', 'R', 'R', 'U', 'U', 'U', 'L', 'D', 'D', 'R', 'D', 'L', 'R', 'U', 'U', 'L', 'L', 'D', 'R', 'D', 'L', 'R', 'R', 'U', 'U', 'L', 'L', 'L', 'D', 'R', 'D', 'L', 'U', 'L', 'U', 'U']", " ['L', 'D', 'D', 'L', 'L', 'L', 'U', 'U', 'U', 'R', 'R', 'D', 'D', 'R', 'D', 'L', 'L', 'U', 'U', 'U', 'R', 'U', 'L']", 
                 "['L', 'U', 'L', 'D', 'D', 'R', 'D', 'L', 'L', 'U', 'U', 'U', 'R', 'U', 'L']", "['L', 'U', 'L', 'D', 'D', 'L', 'U', 'U', 'R', 'D', 'D', 'R', 'D', 'L', 'L', 'U', 'U', 'U', 'R', 'U', 'L']"]  
}

df = pd.DataFrame(data)
print(df)

For map 1:
     Algorithm  Execution Time (seconds)  Visited States  \
0          BFS                    0.0000              47   
1          DFS                    0.0000              17   
2          IDS                    0.0099              17   
3           A*                    0.0040              47   
4  Weighted A*                    0.0030              23   

                              Solution  
0  ['U', 'D', 'L', 'R', 'R', 'L', 'D']  
1  ['D', 'U', 'R', 'L', 'L', 'R', 'U']  
2  ['D', 'U', 'R', 'L', 'L', 'R', 'U']  
3  ['D', 'U', 'R', 'L', 'L', 'R', 'U']  
4  ['U', 'D', 'L', 'R', 'R', 'L', 'D']  
For map 2:
     Algorithm  Execution Time (seconds)  Visited States  \
0          BFS                     0.000              26   
1          DFS                     0.000              13   
2          IDS                     0.000              13   
3           A*                     0.000              27   
4  Weighted A*                     0.001              17   

          